In [1]:
###CHANGELOGG!!!!!

# errorytpe > relative
# objective > multiobjective
# run_BUSBOI >  not bedthenQ, not solver

#version D

#######
# single spline
#monotonics from 0.85 to 0.5
#rmese from 1 to 2



In [2]:
suppressMessages({
    library(dplyr)
    library(parallel)
    library(ggplot2)
    library(tidyr)
    library(hydroGOF)
    })

In [8]:
gauge_df=readRDS('/nas/cee-water/cjgleason/colin/analyze confluence runs/SVS_df.rds')
gauged_reaches=unique(gauge_df$reach_id)
# swot_base='/nas/cee-water/cjgleason/ellie/SWOT/confluence/confluence_relPermML/relPermML_mnt/input/swot/'
# sos_base='/nas/cee-water/cjgleason/ellie/SWOT/confluence/confluence_relPermML/relPermML_mnt/input/sos/'
swot_base='/nas/cee-ice/data/Confluence_Runs/global_vD/global_vD_mnt/input/swot/'
sos_base='/nas/cee-ice/data/Confluence_Runs/global_vD/global_vD_mnt/input/sos/'

reach_ids=gauged_reaches[gauged_reaches %in% substr(list.files(swot_base),1,11)]

In [16]:
output_path='/nas/cee-water/cjgleason/colin/BUSBOI/debug tests/monthly_single_splineD/'
run_ids=substr(list.files(output_path),1,11)
unrun= reach_ids[!reach_ids %in% run_ids]
length(unrun)

[1] 2194

In [15]:
# #save a test case for AWS testing
# #needs:
#     #SWORD
#     #Priors
#     #SWOT input data
# this_reach_id='56424600151'
# sword='/nas/cee-ice/data/SWORD/SWORDv16/netcdf/oc_sword_v16.nc'
# priors=paste0(sos_base,'oc_sword_v16_SOS_priors.nc')
# swot=paste0(swot_base,this_reach_id,'_SWOT.nc')

In [5]:
# versionD=as.data.frame(readRDS('/nas/cee-water/cjgleason/colin/SWOT_global_Q_paper/SES_dataframe_versionD.rds'))%>%
#     select(-geometry)
# versionC=as.data.frame(readRDS('/nas/cee-water/cjgleason/colin/SWOT_global_Q_paper/SES_dataframe.rds'))%>%
#     select(-geometry)

In [5]:
# nrow(filter(versionD,model=='ML_ensemble'))
# nrow(filter(versionC,model=='ML_ensemble'))

In [7]:

# readRDS(list.files('/nas/cee-water/cjgleason/colin/SWOT_global_Q_paper/daily_ensembles_versionD/',full.names=TRUE)[31234])

In [14]:
 source('/nas/cee-water/cjgleason/colin/BUSBOI/BUSBOI/main_function.R')
# suppressWarnings({
# for (i in 19:26){

#     unrun=reach_ids
#     print(i)
#     print(unrun[i])

    
    
# test= main_function(unrun[i],
#                    output_path=output_path,
#                    swot_base=swot_base,
#                    sos_base=sos_base,
#                    Q_prior='daily', #'daily' or 'monthly'
#                    tulip='OFF', #'ON' or 'OFF'
#                    GVF_on=0, # 0 or 1
#                    fix_bed=0) # 0 = 5 pt, 1 = 1pt, 2 = fixed
    
# }

# })

a= Sys.time()
clust=makeCluster(40)
test=parLapply(clust,unrun[1:40],main_function,
                   output_path=output_path,
                   swot_base=swot_base,
                   sos_base=sos_base,
                   Q_prior='monthly', #or 'monthly'
                   tulip='OFF', #or 'OFF'
                   GVF_on=0,
                   fix_bed=0) # 0 = 5 pt, 1 = 1pt, 2 = fixed
stopCluster(clust)
print(Sys.time()-a)

In [17]:
###to control where the slurm files are written
working_dir='/nas/cee-water/cjgleason/colin/BUSBOI/debug_logs/'
setwd(working_dir)

source('/nas/cee-water/cjgleason/colin/BUSBOI/BUSBOI/main_function.R')

library(rslurm, lib.loc = "/nas/cee-water/cjgleason/r-lib/",quietly = TRUE)
library(whisker, lib.loc = "/nas/cee-water/cjgleason/r-lib/",quietly = TRUE)

testname='daily'
#slurm block
slurm_options= list(mem=64000, 'time'='50:00:00', #options for memory, time, parition, and an error file
                    partition ='ceewater_cjgleason-cpu',
                    error='slurm-%A_%a.err')
                    # nodelist = 'ceewater-cpu008')
                    # exclude='ceewater-cpu009')
sjob <- slurm_map(as.list(unrun), #thing you want to loop over. must be a list
                  main_function,  # name of the function. declared above with the 'source' command
                  jobname = testname, # defined above. just for convenience
                   output_path=output_path,
                   swot_base=swot_base,
                   sos_base=sos_base,
                  Q_prior='monthly',
                  tulip='OFF',
                  GVF_on=0,
                  fix_bed=0,  #0 = 5pts, 1= 1pt, 2 = fixed
                  nodes = 1, # many nodes do you want?
                  preschedule_cores=FALSE, # keep this FALSE
                  cpus_per_node = 30, # how many CPUS per node. 
                  submit = TRUE, # if TRUE, submits to the cluster
                  slurm_options=slurm_options, #defined above
                  libPaths="/nas/cee-water/cjgleason/r-lib/" ) #library paths

Submitted batch job 53892228



In [23]:
stopCluster(clust)

In [16]:
#teset busboi  locally
source('/nas/cee-water/cjgleason/colin/Confluence_Offline/debug_testing/modules/busboi/drive_BUSBOI.R')

BUSBOI Configuration
Q Prior Type:      monthly
TULIP:             OFF
GVF Correction:    OFF
Bed Mode:          10-point
Input Directory:   /nas/cee-water/cjgleason/colin/Confluence_Offline/debug_testing/confluence_debug/debug_mnt/input/
Output Directory:  /nas/cee-water/cjgleason/colin/Confluence_Offline/debug_testing/confluence_debug/debug_mnt/flpe/busboi/
[1] "Writing BUSBOI output..."
[1] "Writing posteriors..."
[1] "Writing discharge..."
[1] "Output written to: /nas/cee-water/cjgleason/colin/Confluence_Offline/debug_testing/confluence_debug/debug_mnt/flpe/busboi///73150600131_busboi.nc"
[1] "Writing BUSBOI output..."
[1] "Writing posteriors..."
[1] "Writing discharge..."
[1] "Output written to: /nas/cee-water/cjgleason/colin/Confluence_Offline/debug_testing/confluence_debug/debug_mnt/flpe/busboi///73150600141_busboi.nc"
[1] "Writing BUSBOI output..."
[1] "Writing posteriors..."
[1] "Writing discharge..."
[1] "Output written to: /nas/cee-water/cjgleason/colin/Confluence_Offline/de

In [17]:
files=list.files('/nas/cee-water/cjgleason/colin/Confluence_Offline/debug_testing/confluence_debug/debug_mnt/flpe/busboi/',full.names=TRUE)
for (file in files){
    test=open.nc(file)
print(read.nc(test, recursive=TRUE))
close.nc(test)
    }

$nt
 [1]  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
[26] 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49
[51] 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74
[76] 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96

$nx
 [1] 0 1 2 3 4 5 6 7 8 9

$time_str
 [1] "2023-03-31" "2023-04-02" "2023-04-03" "2023-04-04" "2023-04-05"
 [6] "2023-04-06" "2023-04-07" "2023-04-08" "2023-04-09" "2023-04-10"
[11] "2023-04-11" "2023-04-12" "2023-04-13" "2023-04-14" "2023-04-15"
[16] "2023-04-16" "2023-04-17" "2023-04-18" "2023-04-19" "2023-04-20"
[21] "2023-04-21" "2023-04-22" "2023-04-23" "2023-04-24" "2023-04-25"
[26] "2023-04-26" "2023-04-27" "2023-04-28" "2023-04-29" "2023-04-30"
[31] "2023-05-01" "2023-05-02" "2023-05-03" "2023-05-04" "2023-05-05"
[36] "2023-05-06" "2023-05-07" "2023-05-08" "2023-05-09" "2023-05-10"
[41] "2023-05-11" "2023-05-12" "2023-05-13" "2023-05-14" "2023-05-15"
[46] "2023-05-